In [1]:
import pandas as pd
#from pandas.io.parsers import ParserError
import numpy as np
from helper import get_mapper
import json
import os
import re

In [2]:
from os import listdir, stat
from os.path import isfile, join
BASE_DIR = "."
MIN_SIZE = 512

In [3]:
FOLDERS = [os.path.join(BASE_DIR, o) for o in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR,o))]
FOLDERS.sort()
FOLDERS = FOLDERS[1:-4]
YEARS = ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

In [4]:
FOLDERS

['./2015',
 './2016',
 './2017',
 './2018',
 './2019',
 './2020',
 './2021',
 './2022',
 './2023',
 './2024',
 './2025']

In [5]:
#bpm = pd.read_csv("../basic/block_plant_mapper.csv")
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
#ssem = pd.read_csv("../basic/plant_sse_mapper.csv")
#ssem['plantid'] = ssem['plantid'].apply(lambda x: str(x).replace('/', '_'))

raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

plantslist = list(set(seem['plantid'].to_list()))

/tmp/ipykernel_1467851/4204571207.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))


In [6]:
seem

,plantid,sseid
0,06-02-B10117A007,SEE987197130805
1,06-02-B10117A007,SEE913896693631
2,06-05-100-0030723,BNA1084
3,06-05-100-0431554,BNA0992
4,06-05-100-0431554,BNA0991
...,...,...
1105,SD662-98,SEE927542617698
1106,SD662-98,SEE915847711449
1107,SD662-98,SEE912980761904
1108,SD662-98,SEE918332839635


In [7]:
#plantslist

In [8]:
def save2combined(df, plantname):
    df.to_csv("./combined/" + plantname + ".csv", index=False)

In [9]:
def squash_production(plantslist):
    emptyplants = []
    for plant in plantslist:
        #print(plant)
        dflist = []
        for year in YEARS:
            try:
                df = pd.read_csv("./by_plantid/" + year + "/" + plant + ".csv")
                #df.fillna(0, inplace=True)
                df[df.columns[1:]] = df[df.columns[1:]].astype(int)
            except FileNotFoundError:
                #print("./by_plantid/" + year + "/" + plant + ".csv")
                df = pd.DataFrame({'produced_at': []})
    
            testdf = df.dropna()
            if testdf.empty:
                emptyplants.append(plant)
            else:
                dflist.append(df)
                
        if len(dflist) > 0:
            newdf = pd.concat(dflist)
            save2combined(newdf, plant)

    return list(set(emptyplants))

In [10]:
a = squash_production(plantslist)

In [11]:
len(plantslist)

355

In [12]:
len(a)

332

In [13]:
errorset = []
for plant in plantslist:
    dflist = []
    for year in YEARS:
        df = pd.DataFrame()
        try:
            df = pd.read_csv("./by_plantid/" + year + "/" + plant + ".csv", na_values=['-'])
        except FileNotFoundError:
            if (plant not in errorset):
                errorset.append(plant)
            pass
        
        dflist.append(df)
    newdf = pd.concat(dflist)
    #print(newdf.shape)
print(errorset)

['HE20000115', 'BWpf-450-2951606-00000000', 'NW100-9021016', 'DE.EEA46609', 'BWpf-450-4124448-00000000', 'RP5000656', 'BYS00869', 'NW300-0079450', 'SH50082038', 'NW300-0181417', 'HE50002167', 'HB06-04-11_2000122_0_0', 'ST100032', 'NW500-9981505', 'ST100876', 'SD662-18', 'SD661-85', 'NI31000005008', 'SD661-98', 'BYS00035', 'SD661-94', 'BYS00323', 'ST100850', 'SD661-82', 'NI06277201660', 'BYS00104', 'ST100344', 'MV60004935', 'NW300-0054298', 'MV90031941', 'BWpf-450-3042693-00000000', 'NW100-0030712', 'ST102622', 'HE30001045', 'SD666-13', 'NW700-0020290', 'NW500-0338944', 'SD666-11', 'HE20000004', 'BWpf-450-2719371-00000000', 'NW900-0021170', 'SD666-15', 'NW300-0215520', 'NI01211092310', 'NI05050072350', 'SD661-109', 'BE222276', 'NI10100029340', 'DE.EEA44484', 'SD661-65', 'NW500-0875785', 'ST18046', 'BWpf-450-34064722-00000000', 'HH12273', 'SD661-12', 'NI09261732970', 'SD661-107', 'SD661-39', 'NW100-0387357', 'RP5000031', 'NI04224004080', 'SH10000140', 'ST100609', 'NI01227041760', 'SD661-

In [14]:
newdf

,produced_at,SEE944567587799
0,2015-01-01 00:00:00,0
1,2015-01-01 01:00:00,0
2,2015-01-01 02:00:00,0
3,2015-01-01 03:00:00,0
4,2015-01-01 04:00:00,0
...,...,...
8755,2023-12-31 19:00:00,0
8756,2023-12-31 20:00:00,0
8757,2023-12-31 21:00:00,0
8758,2023-12-31 22:00:00,0


In [15]:
True if '06-05-500-0915123' in errorset else False

False